<a href="https://colab.research.google.com/github/newazkhn/FloodPINN/blob/main/notebooks/06_physics_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 06_physics_uncertainty.ipynb (Improved Version)
# ============================================================
# CHANGES FROM ORIGINAL:
#   1. DATA_FOLDER → FloodPINN_ImprovedChips
#   2. epochs → 50
#   3. encoder → resnet34
#   4. HDF5 files → improved_*.h5
#   5. Model saved as floodpinn_improved_best.pth
# ============================================================

!pip install segmentation-models-pytorch -q
!pip install h5py -q

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import segmentation_models_pytorch as smp
import numpy as np
import h5py
import os
import gc
import random
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import psutil

def get_ram():
    r = psutil.virtual_memory()
    return r.used/1e9, r.total/1e9, r.percent

def get_gpu():
    if torch.cuda.is_available():
        return (
            torch.cuda.memory_allocated()/1e9,
            torch.cuda.get_device_properties(
                0).total_memory/1e9)
    return 0, 0

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 55)
print("  FloodPINN — Improved Retrain")
print("  ResNet34 + Sen1Floods11 + 50 epochs")
print("=" * 55)
print(f"\n  Device : {device}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(
        0).total_memory
    print(f"  VRAM   : {mem/1e9:.1f} GB")

# Paths
DRIVE_BASE    = '/content/drive/MyDrive'

# ← CHANGE 1: Updated data folder
DATA_FOLDER   = (
    f'{DRIVE_BASE}/FloodPINN_ImprovedChips')

MODEL_FOLDER  = f'{DRIVE_BASE}/FloodPINN_Models'
OUTPUT_FOLDER = f'{DRIVE_BASE}/FloodPINN_Results'
os.makedirs(MODEL_FOLDER,  exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Configuration
CONFIG = {
    'model_name'    : 'floodpinn_improved',
    'encoder'       : 'resnet34',    # ← CHANGE 3
    'in_channels'   : 8,
    'out_classes'   : 1,
    'batch_size'    : 4,
    'epochs'        : 50,            # ← CHANGE 2
    'lr'            : 1e-4,
    'weight_decay'  : 1e-5,
    'chip_size'     : 512,
    'seed'          : 42,
    'physics_weight': 0.1,
    'dropout_rate'  : 0.3,
    'n_mc_samples'  : 20,
}

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# ← CHANGE 4: Updated HDF5 filenames
print(f"\n  Checking improved HDF5 files:")
for split in ['train', 'val', 'test']:
    path = f'{DATA_FOLDER}/improved_{split}.h5'
    if os.path.exists(path):
        size = os.path.getsize(path)/1e9
        with h5py.File(path, 'r') as hf:
            n = len(hf['X'])
        print(f"  ✓ improved_{split}.h5 — "
              f"{n} chips — {size:.2f} GB")
    else:
        print(f"  ✗ improved_{split}.h5 NOT FOUND")

ram_used, ram_total, ram_pct = get_ram()
print(f"\n  RAM : {ram_used:.1f}/{ram_total:.1f} GB")
print("\n✓ Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.8 MB/s eta 0:00:00
Mounted at /content/drive
  FloodPINN — Improved Retrain
  ResNet34 + Sen1Floods11 + 50 epochs

  Device : cuda
  GPU    : Tesla T4
  VRAM   : 15.6 GB

  Checking improved HDF5 files:
  ✓ improved_train.h5 — 2177 chips — 10.68 GB
  ✓ improved_val.h5 — 622 chips — 3.07 GB
  ✓ improved_test.h5 — 312 chips — 1.53 GB

  RAM : 1.5/13.6 GB

✓ Setup complete


In [2]:
# ============================================================
# Cell 2 — Dataset and DataLoaders
# ============================================================

class FloodDataset(Dataset):
    """
    Memory efficient HDF5 dataset.
    Reads one chip at a time — never loads full data.
    """

    def __init__(self, h5_path, augment=False):
        self.h5_path = h5_path
        self.augment = augment
        self.h5_file = None
        with h5py.File(h5_path, 'r') as hf:
            self.length = len(hf['X'])

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(
                self.h5_path, 'r')
        X = self.h5_file['X'][idx].astype(
            np.float32)
        y = self.h5_file['y'][idx].astype(
            np.float32)
        X = torch.from_numpy(X)
        y = torch.from_numpy(y)

        if self.augment:
            if random.random() > 0.5:
                X = torch.flip(X, dims=[2])
                y = torch.flip(y, dims=[1])
            if random.random() > 0.5:
                X = torch.flip(X, dims=[1])
                y = torch.flip(y, dims=[0])
            # Extra augmentation — random 90deg rotation
            if random.random() > 0.5:
                k = random.randint(1, 3)
                X = torch.rot90(X, k, dims=[1, 2])
                y = torch.rot90(y, k, dims=[0, 1])

        y = y.unsqueeze(0)
        return X, y

    def close(self):
        if self.h5_file is not None:
            self.h5_file.close()
            self.h5_file = None

    def __del__(self):
        self.close()

# ← CHANGE 4: improved_*.h5 filenames
train_dataset = FloodDataset(
    f'{DATA_FOLDER}/improved_train.h5',
    augment=True)
val_dataset   = FloodDataset(
    f'{DATA_FOLDER}/improved_val.h5',
    augment=False)
test_dataset  = FloodDataset(
    f'{DATA_FOLDER}/improved_test.h5',
    augment=False)

# RAM-safe loader settings
LOADER_KWARGS = dict(
    batch_size  = CONFIG['batch_size'],
    num_workers = 0,
    pin_memory  = False)

train_loader = DataLoader(
    train_dataset, shuffle=True,
    **LOADER_KWARGS)
val_loader   = DataLoader(
    val_dataset,   shuffle=False,
    **LOADER_KWARGS)
test_loader  = DataLoader(
    test_dataset,  shuffle=False,
    **LOADER_KWARGS)

X_s, y_s = next(iter(train_loader))
ram_used, _, ram_pct = get_ram()

print("Datasets ready:")
print(f"  Train : {len(train_dataset)} chips "
      f"({len(train_loader)} batches)")
print(f"  Val   : {len(val_dataset)} chips "
      f"({len(val_loader)} batches)")
print(f"  Test  : {len(test_dataset)} chips "
      f"({len(test_loader)} batches)")
print(f"\nBatch shapes:")
print(f"  X : {X_s.shape}")
print(f"  y : {y_s.shape}")
flood_pct = (y_s==1).float().mean()*100
print(f"  Flood pixels: {flood_pct:.1f}%")
print(f"\nRAM after load: {ram_used:.1f} GB "
      f"({ram_pct:.0f}%)")

del X_s, y_s
gc.collect()
print("\n✓ DataLoaders ready")

Datasets ready:
  Train : 2177 chips (545 batches)
  Val   : 622 chips (156 batches)
  Test  : 312 chips (78 batches)

Batch shapes:
  X : torch.Size([4, 8, 512, 512])
  y : torch.Size([4, 1, 512, 512])
  Flood pixels: 12.3%

RAM after load: 1.8 GB (16%)

✓ DataLoaders ready


In [3]:
# ============================================================
# Cell 3 — MCDropout U-Net with ResNet34
# ← CHANGE 3: encoder = resnet34
# ============================================================

class MCDropoutUNet(nn.Module):
    """
    U-Net with Monte Carlo Dropout.
    ResNet34 encoder — stronger than EfficientNet-b0.
    """

    def __init__(self, encoder='resnet34',
                 in_channels=8, out_classes=1,
                 dropout_rate=0.3):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.unet = smp.Unet(
            encoder_name         = encoder,
            encoder_weights      = 'imagenet',
            in_channels          = in_channels,
            classes              = out_classes,
            activation           = None,
            decoder_use_batchnorm= True,
        )
        self.mc_dropout = nn.Dropout2d(
            p=dropout_rate)

    def forward(self, x):
        out = self.unet(x)
        return self.mc_dropout(out)

    def mc_predict(self, x, n_samples=20):
        """MC Dropout uncertainty estimation."""
        self.train()  # keep dropout active
        predictions = []
        with torch.no_grad():
            for _ in range(n_samples):
                pred = torch.sigmoid(
                    self.forward(x))
                predictions.append(pred)
        all_preds = torch.stack(
            predictions, dim=0)
        mean_pred = all_preds.mean(dim=0)
        std_pred  = all_preds.std(dim=0)
        return mean_pred, std_pred, all_preds

# Build model with ResNet34
model_novel = MCDropoutUNet(
    encoder      = CONFIG['encoder'],
    in_channels  = CONFIG['in_channels'],
    out_classes  = CONFIG['out_classes'],
    dropout_rate = CONFIG['dropout_rate']
)
model_novel = model_novel.to(device)

# Verify forward pass
with torch.no_grad():
    test_in  = torch.randn(
        1, CONFIG['in_channels'],
        CONFIG['chip_size'],
        CONFIG['chip_size']).to(device)
    test_out = model_novel(test_in)
    del test_in, test_out
    torch.cuda.empty_cache()

total_params = sum(
    p.numel() for p in model_novel.parameters())

ram_used, _, ram_pct = get_ram()
gpu_used, gpu_total  = get_gpu()

print(f"Novel Model — ResNet34 + MC Dropout:")
print(f"  Encoder      : {CONFIG['encoder']}")
print(f"  Dropout rate : {CONFIG['dropout_rate']}")
print(f"  MC samples   : {CONFIG['n_mc_samples']}")
print(f"  Total params : {total_params:,}")
print(f"\nMemory:")
print(f"  CPU RAM : {ram_used:.1f} GB "
      f"({ram_pct:.0f}%)")
print(f"  GPU RAM : {gpu_used:.1f}/"
      f"{gpu_total:.1f} GB")
print("\n✓ Model ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Novel Model — ResNet34 + MC Dropout:
  Encoder      : resnet34
  Dropout rate : 0.3
  MC samples   : 20
  Total params : 24,452,049

Memory:
  CPU RAM : 2.1 GB (18%)
  GPU RAM : 0.1/15.6 GB

✓ Model ready


In [4]:
# ============================================================
# Cell 4 — Physics Loss + Metrics
# ============================================================

class PhysicsInformedLoss(nn.Module):
    """
    Focal + Dice + Physics terrain constraint.
    Physics: water flows to low elevation areas.
    DEM channel index = 5.
    """

    def __init__(self, focal_weight=0.45,
                 dice_weight=0.45,
                 physics_weight=0.10,
                 dem_channel=5):
        super().__init__()
        self.focal_weight   = focal_weight
        self.dice_weight    = dice_weight
        self.physics_weight = physics_weight
        self.dem_channel    = dem_channel

    def focal_loss(self, pred, target,
                   alpha=0.8, gamma=2.0):
        bce   = F.binary_cross_entropy_with_logits(
            pred, target, reduction='none')
        pt    = torch.exp(-bce)
        focal = alpha*((1-pt)**gamma)*bce
        return focal.mean()

    def dice_loss(self, pred, target,
                  smooth=1e-6):
        pred   = torch.sigmoid(pred).view(-1)
        target = target.view(-1)
        inter  = (pred * target).sum()
        return 1 - (2*inter + smooth) / (
            pred.sum() + target.sum() + smooth)

    def physics_loss(self, pred, inputs):
        dem = inputs[
            :, self.dem_channel:
               self.dem_channel+1, :, :]
        dem_min = dem.flatten(2).min(
            dim=2)[0].unsqueeze(-1).unsqueeze(-1)
        dem_max = dem.flatten(2).max(
            dim=2)[0].unsqueeze(-1).unsqueeze(-1)
        dem_norm = (dem - dem_min) / (
            dem_max - dem_min + 1e-8)
        flood_prob = torch.sigmoid(pred)
        return (dem_norm * flood_prob).mean()

    def forward(self, pred, target, inputs):
        focal   = self.focal_loss(pred, target)
        dice    = self.dice_loss(pred, target)
        physics = self.physics_loss(
            pred, inputs)
        total = (self.focal_weight   * focal +
                 self.dice_weight    * dice  +
                 self.physics_weight * physics)
        return total, {
            'focal'  : focal.item(),
            'dice'   : dice.item(),
            'physics': physics.item(),
            'total'  : total.item()
        }


def compute_metrics(pred_logits, target,
                    threshold=0.5):
    """IoU, F1, Precision, Recall."""
    pred   = (torch.sigmoid(pred_logits)
              > threshold).float()
    target = target.float()
    pred   = pred.view(-1)
    target = target.view(-1)

    smooth    = 1e-6
    TP = (pred * target).sum()
    FP = (pred * (1-target)).sum()
    FN = ((1-pred) * target).sum()

    precision = TP / (TP + FP + smooth)
    recall    = TP / (TP + FN + smooth)
    f1        = (2 * precision * recall /
                 (precision + recall + smooth))
    iou       = TP / (TP + FP + FN + smooth)

    return {
        'iou'      : iou.item(),
        'f1'       : f1.item(),
        'precision': precision.item(),
        'recall'   : recall.item(),
    }


# Initialise loss and optimiser
criterion = PhysicsInformedLoss(
    focal_weight   = 0.45,
    dice_weight    = 0.45,
    physics_weight = CONFIG['physics_weight'])

optimizer = optim.Adam(
    model_novel.parameters(),
    lr           = CONFIG['lr'],
    weight_decay = CONFIG['weight_decay'])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode    = 'min',
    factor  = 0.5,
    patience= 5)   # increased from 3

print("Loss and optimiser ready:")
print(f"  Focal weight   : 0.45")
print(f"  Dice weight    : 0.45")
print(f"  Physics weight : "
      f"{CONFIG['physics_weight']}")
print(f"  Optimiser : Adam lr={CONFIG['lr']}")
print(f"  Scheduler : ReduceLROnPlateau "
      f"patience=5")
print("\n✓ Ready to train")

Loss and optimiser ready:
  Focal weight   : 0.45
  Dice weight    : 0.45
  Physics weight : 0.1
  Optimiser : Adam lr=0.0001
  Scheduler : ReduceLROnPlateau patience=5

✓ Ready to train


In [ ]:
# ============================================================
# Cell 5 — Training Loop (50 epochs)
# ← CHANGE 5: saves to floodpinn_improved_best.pth
# ============================================================

def train_epoch_physics(model, loader,
                        criterion, optimizer,
                        device):
    model.train()
    total_loss     = 0
    loss_breakdown = {
        'focal':0, 'dice':0, 'physics':0}
    all_metrics    = {
        'iou':0, 'f1':0,
        'precision':0, 'recall':0}
    n_batches = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss, breakdown = criterion(pred, y, X)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        for k in loss_breakdown:
            loss_breakdown[k] += breakdown[k]
        m = compute_metrics(pred.detach(), y)
        for k in all_metrics:
            all_metrics[k] += m[k]
        n_batches += 1

        del X, y, pred, loss

    torch.cuda.empty_cache()
    gc.collect()

    return (total_loss / n_batches,
            {k: v/n_batches
             for k, v in all_metrics.items()},
            {k: v/n_batches
             for k, v in loss_breakdown.items()})


def val_epoch_physics(model, loader,
                      criterion, device):
    model.eval()
    total_loss  = 0
    all_metrics = {
        'iou':0, 'f1':0,
        'precision':0, 'recall':0}
    n_batches   = 0

    with torch.no_grad():
        for X, y in loader:
            X    = X.to(device, non_blocking=True)
            y    = y.to(device, non_blocking=True)
            pred = model(X)
            loss, _ = criterion(pred, y, X)

            total_loss += loss.item()
            m = compute_metrics(pred, y)
            for k in all_metrics:
                all_metrics[k] += m[k]
            n_batches += 1
            del X, y, pred, loss

    torch.cuda.empty_cache()
    gc.collect()

    return (total_loss / n_batches,
            {k: v/n_batches
             for k, v in all_metrics.items()})


# Training setup
history_novel = {
    'train_loss'  : [], 'val_loss'    : [],
    'train_iou'   : [], 'val_iou'     : [],
    'train_f1'    : [], 'val_f1'      : [],
    'physics_loss': [],
}

best_val_iou_novel = 0.0
best_val_f1_novel  = 0.0

# ← CHANGE 5: new save path
best_model_path = (
    f'{MODEL_FOLDER}/'
    f'floodpinn_improved_best.pth')

BASELINE_IOU = 0.1998  # previous best
BASELINE_F1  = 0.2788  # previous best

print("=" * 65)
print("  Training FloodPINN — Improved Version")
print("=" * 65)
print(f"  Encoder        : {CONFIG['encoder']}")
print(f"  Training chips : {len(train_dataset)}")
print(f"  Val chips      : {len(val_dataset)}")
print(f"  Epochs         : {CONFIG['epochs']}")
print(f"  Baseline IoU   : {BASELINE_IOU}")
print(f"  Baseline F1    : {BASELINE_F1}")

ram_used, ram_total, _ = get_ram()
print(f"\n  RAM : {ram_used:.1f}/{ram_total:.1f} GB")

print(f"\n{'Ep':>3} | {'TLoss':>7} | "
      f"{'VLoss':>7} | {'TIoU':>6} | "
      f"{'VIoU':>6} | {'VF1':>6} | "
      f"{'Phys':>6} | {'RAM':>5}")
print("-" * 72)

for epoch in range(1, CONFIG['epochs']+1):

    train_loss, train_m, breakdown = \
        train_epoch_physics(
            model_novel, train_loader,
            criterion, optimizer, device)

    val_loss, val_m = val_epoch_physics(
        model_novel, val_loader,
        criterion, device)

    scheduler.step(val_loss)

    # Save history
    history_novel['train_loss'].append(train_loss)
    history_novel['val_loss'].append(val_loss)
    history_novel['train_iou'].append(
        train_m['iou'])
    history_novel['val_iou'].append(val_m['iou'])
    history_novel['train_f1'].append(train_m['f1'])
    history_novel['val_f1'].append(val_m['f1'])
    history_novel['physics_loss'].append(
        breakdown['physics'])

    # Save best model
    if val_m['iou'] > best_val_iou_novel:
        best_val_iou_novel = val_m['iou']
        best_val_f1_novel  = val_m['f1']
        torch.save({
            'epoch'      : epoch,
            'model_state': model_novel.state_dict(),
            'optimizer'  : optimizer.state_dict(),
            'val_iou'    : best_val_iou_novel,
            'val_f1'     : best_val_f1_novel,
            'config'     : CONFIG,
            'history'    : history_novel,
        }, best_model_path)
        saved = "★"
    else:
        saved = ""

    # Beat baseline indicator
    beat = ""
    if val_m['iou'] > BASELINE_IOU:
        beat = "← BEAT!"
    if val_m['f1'] > 0.50:
        beat = "← F1>0.50!"
    if val_m['f1'] > 0.60:
        beat = "← F1>0.60! ✓"

    ram_used, _, ram_pct = get_ram()

    print(f"{epoch:>3} | {train_loss:>7.4f} | "
          f"{val_loss:>7.4f} | "
          f"{train_m['iou']:>6.4f} | "
          f"{val_m['iou']:>6.4f} | "
          f"{val_m['f1']:>6.4f} | "
          f"{breakdown['physics']:>6.4f} | "
          f"{ram_used:>4.1f}G "
          f"{saved} {beat}")

print(f"\n{'='*65}")
print(f"Training complete!")
print(f"  Best Val IoU : {best_val_iou_novel:.4f}")
print(f"  Best Val F1  : {best_val_f1_novel:.4f}")
print(f"  vs Baseline IoU : {BASELINE_IOU} "
      f"{'✓ IMPROVED' if best_val_iou_novel > BASELINE_IOU else '✗'}")
print(f"  vs Baseline F1  : {BASELINE_F1} "
      f"{'✓ IMPROVED' if best_val_f1_novel > BASELINE_F1 else '✗'}")
print(f"\n  Model saved: {best_model_path}")

NameError: name 'MODEL_FOLDER' is not defined

In [6]:
# ============================================================
# Cell 6 — Test evaluation + uncertainty maps
# ============================================================

# Define path — session restarted so variable lost
best_model_path = (
    f'{MODEL_FOLDER}/'
    f'floodpinn_improved_best.pth')

# Load best model
checkpoint = torch.load(
    best_model_path, map_location=device)
model_novel.load_state_dict(
    checkpoint['model_state'])
print(f"Best model from epoch "
      f"{checkpoint['epoch']}")
print(f"  Val IoU : {checkpoint['val_iou']:.4f}")
print(f"  Val F1  : {checkpoint['val_f1']:.4f}")

best_model_path = (
    f'{MODEL_FOLDER}/'
    f'floodpinn_improved_best.pth')

# Load best model
checkpoint = torch.load(
    best_model_path, map_location=device)
model_novel.load_state_dict(
    checkpoint['model_state'])
print(f"Best model from epoch "
      f"{checkpoint['epoch']}")
print(f"  Val IoU : {checkpoint['val_iou']:.4f}")
print(f"  Val F1  : {checkpoint['val_f1']:.4f}")

# Test evaluation
model_novel.eval()
test_metrics = {
    'iou':0, 'f1':0,
    'precision':0, 'recall':0}
n_batches = 0

with torch.no_grad():
    for X, y in test_loader:
        X    = X.to(device)
        y    = y.to(device)
        pred = model_novel(X)
        m    = compute_metrics(pred, y)
        for k in test_metrics:
            test_metrics[k] += m[k]
        n_batches += 1
        del X, y, pred

torch.cuda.empty_cache()
test_metrics = {k: v/n_batches
                for k, v in test_metrics.items()}

print(f"\n{'='*55}")
print(f"  FloodPINN Improved — Test Results")
print(f"{'='*55}")
print(f"  IoU       : {test_metrics['iou']:.4f}"
      f"  (baseline: 0.1998)")
print(f"  F1        : {test_metrics['f1']:.4f}"
      f"  (baseline: 0.2788)")
print(f"  Precision : "
      f"{test_metrics['precision']:.4f}")
print(f"  Recall    : "
      f"{test_metrics['recall']:.4f}")

iou_imp = ((test_metrics['iou'] - 0.1998)
           / 0.1998 * 100)
f1_imp  = ((test_metrics['f1']  - 0.2788)
           / 0.2788 * 100)
print(f"\n  IoU improvement: {iou_imp:+.1f}%")
print(f"  F1  improvement: {f1_imp:+.1f}%")

# Save results
np.save(
    f'{OUTPUT_FOLDER}/improved_results.npy',
    test_metrics)
print(f"\n✓ Results saved")

Best model from epoch 45
  Val IoU : 0.3202
  Val F1  : 0.4556
Best model from epoch 45
  Val IoU : 0.3202
  Val F1  : 0.4556

  FloodPINN Improved — Test Results
  IoU       : 0.3790  (baseline: 0.1998)
  F1        : 0.5151  (baseline: 0.2788)
  Precision : 0.4805
  Recall    : 0.6229

  IoU improvement: +89.7%
  F1  improvement: +84.8%

✓ Results saved


In [7]:
# ============================================================
# Fix widget metadata — run before saving to GitHub
# ============================================================

import json
import nbformat

# Clear widget state from current notebook
# This fixes the GitHub rendering error
try:
    from google.colab import output
    output.clear()
except:
    pass

print("✓ Ready to save to GitHub")
print("  File → Save a copy in GitHub")

✓ Ready to save to GitHub
  File → Save a copy in GitHub


In [8]:
# ============================================================
# Fix GitHub rendering error — run this LAST before saving
# ============================================================

import json
import IPython

def fix_notebook_widgets():
    """Remove broken widget metadata from notebook."""
    try:
        # Get the current notebook path
        import ipykernel
        from pathlib import Path

        # Clear widget metadata via JavaScript
        display_js = """
        (function() {
            var nb = Jupyter.notebook;
            if (nb && nb.metadata && nb.metadata.widgets) {
                delete nb.metadata.widgets;
                nb.save_notebook();
                console.log('Widget metadata removed');
            }
        })();
        """
        IPython.display.display(
            IPython.display.Javascript(display_js))
        print("✓ Widget metadata cleared")
    except Exception as e:
        print(f"Note: {e}")

fix_notebook_widgets()
print("Now save: File → Save a copy in GitHub")

<IPython.core.display.Javascript object>

✓ Widget metadata cleared
Now save: File → Save a copy in GitHub
